## Importing Libraries

In [34]:
import glob
import json
from tqdm import tqdm
import random
import os
from groq import Groq

## Setup Files

In [35]:
GROQ_KEY = os.getenv("GROQ_API_KEY")
PATIENT_PROFILES = glob.glob("./patient_profiles/patient_*.json")
GEN_PROMPT = "./prompts/diary_generation_prompt.txt"
SYS_PROMPT = "./prompts/system_prompt.txt"
DIARY_TEMPLATE = "./diaries_template/diary_template.txt"
DIARY_EXAMPLES = "./diaries_template/diaries_ex.txt"

OUTPUT_DIR = "./outputs/"
OUTPUT_EXP_DIR = "./outputs/diary-gen_experiment"
OUTPUT_FILE = "diary-gen_patient"

## Setup Environment

In [36]:
## Setting evironment
os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isdir(os.path.join(OUTPUT_DIR, path)):
        count += 1
        
os.makedirs(f"{OUTPUT_EXP_DIR}_{count}", exist_ok=True)

## Generating Diaries

In [37]:
client = Groq(api_key=GROQ_KEY)

pbar = tqdm(total=len(PATIENT_PROFILES), desc="Generating sythentic clinical diaries")

for patient in PATIENT_PROFILES:
    print("Processing patient:", patient)
    with open(patient, "r", encoding="utf-8") as f, \
         open(GEN_PROMPT, "r", encoding="utf-8") as gen_prompt_file, \
         open(SYS_PROMPT, "r", encoding="utf-8") as sys_prompt_file, \
         open(DIARY_TEMPLATE, "r", encoding="utf-8") as diary_template_file, \
         open(DIARY_EXAMPLES, "r", encoding="utf-8") as diary_ex_file:
             
        patient_id = patient.split('_')[2].split('.')[0]
            
        patient_data = json.load(f)
        base_gen_prompt = gen_prompt_file.read()
        sys_prompt = sys_prompt_file.read()
        diary_template = diary_template_file.read()
        diary_examples = diary_ex_file.read()
        
        prompt_w_template = base_gen_prompt.replace("{{TEMPLATE_TEXT}}", diary_template)
        prompt_w_patient = prompt_w_template.replace("{{PATIENT_PROF}}", json.dumps(patient_data))
        prompt_final = prompt_w_patient.replace("{{DIARIES_TEXT}}", diary_examples)
        
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": sys_prompt
                },
                {
                    "role": "user",
                    "content": prompt_final
                }
            ]
        )
        result = completion.choices[0].message.content
        
        with open(f"{OUTPUT_EXP_DIR}_{count}/{OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
            o.write(f"{result}\n\n")
            print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Generating sythentic clinical diaries:   0%|          | 0/10 [05:20<?, ?it/s]


Processing patient: ./patient_profiles\patient_1.json


Saved LLM output on ./outputs/diary-gen_experiment_0


Processing patient: ./patient_profiles\patient_10.json


Saved LLM output on ./outputs/diary-gen_experiment_0


Processing patient: ./patient_profiles\patient_2.json


Saved LLM output on ./outputs/diary-gen_experiment_0


Processing patient: ./patient_profiles\patient_3.json


...

Saved LLM output on ./outputs/diary-gen_experiment_0


Processing patient: ./patient_profiles\patient_7.json


Saved LLM output on ./outputs/diary-gen_experiment_0


Processing patient: ./patient_profiles\patient_8.json


Saved LLM output on ./outputs/diary-gen_experiment_0


Processing patient: ./patient_profiles\patient_9.json


Generating sythentic clinical diaries: 100%|██████████| 10/10 [03:24<00:00, 20.49s/it]

Saved LLM output on ./outputs/diary-gen_experiment_0


